# C7 · Extracción por exposición y combinación

**Spec:** [`docs/spec_C7_codex_perexp_combine.md`](../docs/spec_C7_codex_perexp_combine.md)  |  **Bloque:** C · Extracción  |  **Run de este set:** `ROXs42Bb_realigned`

Extrae el compañero en CADA exposición con su propia PSF y combina las MEDIDAS, en vez de combinar los cubos y extraer una vez.

| | |
|---|---|
| **Entrada** | `psf_model_mixture.json` de forma `mixture` (C1 con `psf_scope=per_observation`), `observation_plan.json`, `stage01c_qc.json` y los cubos por exposición |
| **Salida (QC/productos)** | `stages/spec_perexp_<grupo>_<apertura>_object.fits` + sus controles `.npz` + `stages/spec_perexp_qc.json` |
| **Consume aguas abajo** | Ninguna etapa: NO entra en `METHOD_ORDER`. Es producto de referencia y validación contra los seis métodos de C2–C6. |


## Qué hace C7 y por qué

La corrección de apertura que consumen C2–C6 sale del modelo de PSF de C1, y ese modelo es una **mezcla** de las N exposiciones. Su error está medido y es cromático: la V4 de la spec C1 falla con +6.77 %, repartido en +0.34 % en el azul y **+8.55 % en el rojo**.

C7 extrae en cada exposición con **su** PSF —donde el modelo vale— y combina las medidas. Lo medido el 2026-08-26: la S/N **empata** con el cubo combinado (0.97–1.09×), pero la razón `apcorr_perexp/apcorr_mezcla` deriva **−8.52 %** del azul al rojo, justo lo contrario que la V4 de la mezcla (+8.2 %).

**Lo que se gana no es S/N, es poder pesar por calidad.** El combinado pesa por `exptime`, y por eso le da el 38.1 % del peso a una noche con σ 4.2× peor.

**Rol en la cadena:** ninguno. No entra en `METHOD_ORDER` —un método nuevo allí se vuelve obligatorio para todos los runs al instante— así que D2 no la calibra y no entra en el bloque E. Es referencia y validación.


## Cómo ejecutar de forma independiente

```bash
conda activate MUSE               # kernel/env con astropy + musepipe
export RUN=ROXs42Bb_realigned   # el run de este objeto
cd MUSE-accretion-pipeline                    # raíz del repo
python -m musepipe.stages.stage_x06_perexp --run-id $RUN
```

Pesado: una pasada de lectura sobre las 29–30 exposiciones (~14 min).

La celda de abajo hace lo mismo desde el notebook (guardada por `RUN`).


In [ ]:
import os, sys
# Localiza la raíz del repo ascendiendo hasta encontrar `musepipe/` (robusto a
# la profundidad: funciona con el cwd en notebooks/<obj>/, en notebooks/ o en la
# raíz). Añade la raíz (para `import musepipe`) y notebooks/ (para `_nbcommon`).
_d = os.getcwd()
while _d != os.path.dirname(_d):
    if os.path.isdir(os.path.join(_d, 'musepipe')) and os.path.isdir(os.path.join(_d, 'notebooks')):
        break
    _d = os.path.dirname(_d)
_root = _d
for _p in (_root, os.path.join(_root, 'notebooks')):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import _nbcommon as nb
try:
    import matplotlib as mpl
    mpl.rcParams['figure.dpi'] = 120     # retina dobla esto sin agrandar
    mpl.rcParams['savefig.dpi'] = 200
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats('retina')
except Exception:
    pass
RUN_ID = nb.resolve_run_id('ROXs42Bb_realigned')
print('run  =', RUN_ID)
print('root =', _root)
print('dir  =', nb.run_dir(RUN_ID))
print('QC   =', nb.provenance_line('stages/spec_perexp_qc.json', RUN_ID))


## Ejecutar o auditar


In [ ]:
RUN = False   # -> True para RE-EJECUTAR esta etapa (regenera su QC)

if RUN:
    cmd = 'python -m musepipe.stages.stage_x06_perexp --run-id $RUN'.replace('$RUN', RUN_ID)
    print('ejecutando:', cmd)
    import subprocess
    subprocess.run(cmd, shell=True, cwd=str(nb.project_root()), check=True)
else:
    print('Modo auditoría (RUN=False): se carga el QC existente abajo.')


## QC / resultados


In [ ]:
qc = nb.load_qc_optional('stages/spec_perexp_qc.json', RUN_ID)
nb.show(qc, keys=['convention.combine', 'convention.group_by', 'convention.weight_band_A', 'input.n_exposures', 'weight_share_by_night'], title='C7')


## Los términos de este QC, en físico

| Término | Qué es | Por qué importa |
|---|---|---|
| `convention.combine` | La ley de pesos con la que se combinaron las medidas. | Es lo que decide el resultado: medido en ROXs 12 b, `invvar` da S/N 19.33 en el continuo y `exptime` —la ley del cubo combinado— da **0.11**. |
| `convention.weight_band_A` | La banda donde se miden los pesos. | Tiene que ser **distinta** de la banda que se mide: pesar con la σ de la propia banda infla la S/N un 40–60 % por auto-selección. |
| `groups[].apertures[].n_eff` | Exposiciones efectivas tras pesar. | 16 de 29 en ROXs 12 b: la mitad del tiempo de telescopio pesa poco, y eso es un hecho del dato. |
| `weight_share_by_night` | Cuánto peso se lleva cada noche con cada ley. | El combinado canónico da **38.1 %** a la peor noche de ROXs 12 b porque pesa por segundos; `invvar` le da **2.6 %**. |


## Resultados que llevaron a la conclusión

Dos vías se cerraron **con medida** antes de llegar aquí: el término híbrido de C1 (arregla el anillo y lleva la V4 de +6.04 % a **+35.08 %**) y cambiar la ley de combinación de la mezcla (media +4.28 %, `sigclip` +4.72 %, mediana −12.38 %).

Y agrupar por noche no es comodidad: con solo las 22 buenas la S/N sube a 20.60 / 9.24 / 401.36 (continuo/Hα/rojo) frente a 19.33 / 8.35 / 388.75 con las 29.


## Decisiones y notas
- No entra en `METHOD_ORDER`: sería obligatoria para todos los runs al instante y arrastraría D1, D2, E1, E3, E4, G1 y F1. · [`docs/spec_C7_codex_perexp_combine.md`](../docs/spec_C7_codex_perexp_combine.md)
- La σ sale de los controles COMBINADOS con los mismos pesos, no de `STAT` ni de σᵢ/√n —esta última trata como ruido la estructura del halo, que no promedia. · [`docs/noise_model.md`](../docs/noise_model.md)
- Los pesos de `invvar` se miden en una banda declarada, nunca en la que se mide: eso último infla la S/N por auto-selección. · [`docs/2026-08-26_perexp_medido_y_la_noche_mala.md`](../docs/2026-08-26_perexp_medido_y_la_noche_mala.md)
- `sum` está declarada y da la MISMA S/N que `equal` (difieren en el factor N): sirve para conservar flujo, no para el mejor límite. · [`docs/spec_C7_codex_perexp_combine.md`](../docs/spec_C7_codex_perexp_combine.md)
